# A custom encrypted model, all from the Python binding

`perseus` runs neural networks under CKKS fully homomorphic encryption on GPU
(FIDESlib + OpenFHE). This notebook defines a **custom model in pure Python** with
torch-style `Enc*` modules, runs it **encrypted end-to-end**, then closes the
production loop: **capture** the operation graph, plan its bootstraps with the min-cut
planner, and run the plan.

In [ ]:
import os

import numpy as np

from perseus import _core
from perseus.nn import EncGELU, EncLinear, EncSequential

opts = _core.InferenceOptions()
opts.ckks = _core.CKKSOptions.from_env()
opts.mode = _core.InferenceMode.Sync
inf = _core.make_gpt2_inference(opts)   # the GPT-2 session: its rotation-key band is larger than a small model needs
print(f"slots={inf.slots}  logN={inf.logN}  level_limit={inf.fhe.level_limit()}")

## Define the model

An encrypted expand/contract MLP (1024 → 4096 → GELU → 1024 padded; 768/3072 real), like a transformer MLP. The GELU is a
*calibration-carrying* module: its polynomial approximation comes from a fitted section
of `configs.json`. Borrowing GPT-2's fit works here because the weight scale below puts
the activations in the same range; for a model of your own, fit the sections to it with
`perseus.nn.calibrate_sequential`.

Weights are plain numpy matrices `(d_in, d_out)`, encoded as CKKS *plaintexts* on the
server at bind time — only the activations are encrypted (see `docs/SECURITY_MODEL.md`).

In [ ]:
CONFIGS = os.environ.get("CONFIGS_PATH",
                         "configs/model/approximation/gpt2_base_n32/configs.json")
configs = _core.load_configs(CONFIGS)   # the shipped calibration; scripts/run_notebooks.sh sets it

d_pad, d_real = 1024, 768
d_exp, e_real = 4096, 3072
rng = np.random.default_rng(0)


def make_weight(d_in, d_out, in_real, out_real, scale=0.5):
    w = rng.standard_normal((d_in, d_out)) * scale / np.sqrt(in_real)
    w[in_real:, :] = 0.0
    w[:, out_real:] = 0.0
    return w


W1 = make_weight(d_pad, d_exp, d_real, e_real)
W2 = make_weight(d_exp, d_pad, e_real, d_real)

# torch style model definition
model = EncSequential(
    EncLinear("fc1", d_pad, d_exp, weight=W1.tolist()),
    EncGELU("act"),
    EncLinear("fc2", d_exp, d_pad, weight=W2.tolist()),
).bind(inf)
inf.set_gelu_cfg("act", configs.softgelu["transformer.h.0.mlp.act"])
print("model bound: fc1(1024->4096) -> gelu -> fc2(4096->1024), weights encoded on device as CKKS plaintexts")

## Encrypted forward vs plaintext reference

Encrypt one input vector, run the model **eagerly** (the runtime fires bootstraps
automatically whenever a ciphertext runs out of levels), and compare against the exact
plaintext computation.

In [ ]:
x_real = rng.standard_normal(d_real) * 0.3


def gelu_exact(v):
    return 0.5 * v * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (v + 0.044715 * v**3)))


x_pad = np.zeros(d_pad)
x_pad[:d_real] = x_real
ref = (gelu_exact(x_pad @ W1) @ W2)[:d_real]   # (d_in, d_out) weights: y = x @ W

x = _core.encode_token_input(inf, x_real.tolist())
y = model(x)
out = np.array(_core.decode_token_output(inf, y))[:d_real]

rel = np.linalg.norm(out - ref) / np.linalg.norm(ref)
print(f"eager encrypted forward: rel_err={rel:.3e}  (in L{x.level} -> out L{y.level})")
assert rel < 0.05

## Capture the operation graph

Every CKKS primitive the model executes is recorded: op type, operands, levels, and
observed magnitudes. The captured graph is the planner's single source of truth.

In [ ]:
graph_dir = ".cache/custom_seq/block_0"   # relative to the repo root (run_notebooks.sh cds there)
os.makedirs(graph_dir, exist_ok=True)

_core.reset_graph_runtime(inf)
inf.enable_graph_capture()
model(_core.encode_token_input(inf, x_real.tolist()))
inf.export_graph_json(f"{graph_dir}/graph.json")
inf.disable_graph_capture()

import json

nodes = json.load(open(f"{graph_dir}/graph.json"))["nodes"]
eager_bts = sum(n["op_type"] == "auto_bootstrap" for n in nodes)
print(f"captured {len(nodes)} nodes, {eager_bts} eager auto-bootstraps")

## Plan: a custom bootstrap schedule

`perseus.plan` runs a range-aware min-cut over the captured graph: it erases the
eager greedy bootstraps and re-places them on magnitude-safe, level-optimal variables.
The output is a placement plan an artifact bound to this exact model.

In [ ]:
from perseus.plan import PlanConfig, plan_graph_dir

# The planner works in the chain's own level units, and PlanConfig's defaults are the
# 64-bit chain's. Take them from the session instead, so this runs on either build.
unit = 2 if _core.chain == "n32" else 1
land = 34 if unit == 2 else 16                      # where a fresh encode and a refresh land
cfg = PlanConfig(bootstrap_level=land, source_level=land, cache_read_level=land,
                 max_level=inf.fhe.level_limit(), level_unit=unit,
                 acc_chain="n32" if unit == 2 else "")

plan_dir = ".cache/custom_seq_plan"
summaries = plan_graph_dir(".cache/custom_seq", plan_dir, cfg)

n_planned = sum(s["num_placements"] for s in summaries)
print(f"planned placements: {n_planned} (eager fired {eager_bts} auto-bootstraps)")

## Planned deterministic rerun

Load the plan and run again. Planned mode is **strict**: every bootstrap fires exactly
where the plan says, and any divergence between model and plan raises instead of
silently degrading.

In [ ]:
_core.reset_graph_runtime(inf)
inf.load_bootstrap_plan_json(f"{plan_dir}/block_0_placement.json")   # returns False for a hint-only plan, which still binds

y_planned = model(_core.encode_token_input(inf, x_real.tolist()))
inf.clear_bootstrap_plan()

out_planned = np.array(_core.decode_token_output(inf, y_planned))[:d_real]
rel_planned = np.linalg.norm(out_planned - ref) / np.linalg.norm(ref)
print(f"planned forward: rel_err={rel_planned:.3e}  placements {eager_bts} -> {n_planned}")
assert rel_planned < 0.05

## Performance

Per-forward wall time (encrypt → forward → decrypt), eager vs planned, on one GPU.
This chain is shallow enough that eager never runs out of levels, so here the plan buys
determinism and the timing shows the raw cost of its placements.
At depth the ranking flips: bootstraps dominate the latency of deep models and the planner places them
deliberately (encrypted GPT-2 decode on the 32-bit chain: eager 37.64 s/token vs planned 15.44, README Table 1).

In [ ]:
import time


def one_forward():
    _core.reset_graph_runtime(inf)   # restart var naming so the strict plan re-binds
    y = model(_core.encode_token_input(inf, x_real.tolist()))
    return _core.decode_token_output(inf, y)


def timeit(n=5):
    one_forward()   # warm
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        one_forward()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts) * 1e3, np.std(ts) * 1e3


eager_ms = timeit()
inf.load_bootstrap_plan_json(f"{plan_dir}/block_0_placement.json")   # returns False for a hint-only plan, which still binds
ms_planned = timeit()
inf.clear_bootstrap_plan()

print(f"eager   forward: {eager_ms[0]:6.0f} ± {eager_ms[1]:.0f} ms   ({eager_bts} eager bootstraps)")
print(f"planned forward: {ms_planned[0]:6.0f} ± {ms_planned[1]:.0f} ms   ({n_planned} planned placements)")
print(f"device memory free: {_core.device_free_gb():.1f} GB")

## What this showed

- A model **defined only in Python** (torch-style modules, numpy weights) ran fully
  encrypted on the CUDA FHE runtime.
- The **capture → plan → planned-strict** production loop closed entirely from Python:
  the same planner that produced the GPT-2 plans binds custom models.
- Encrypted-forward latency measured directly, eager vs planned, on one GPU.